# 03 — Feature engineering

In [2]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

from fraud.data import load_train
from fraud.split import time_split

In [3]:
df = load_train()
train, valid, _ = time_split(df)

In [4]:
NON_FEATURES = ["isFraud", "TransactionDT", "TransactionID"]

MODEL_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.5,
    n_jobs=8,
    verbose=-1,
)

def evaluate(train, valid, seed=42):
    """Fit on train, score on valid."""
    model = lgb.LGBMClassifier(**MODEL_PARAMS,random_state=seed)
    model.fit(train.drop(columns=NON_FEATURES), train['isFraud'])
    scores = model.predict_proba(valid.drop(columns=NON_FEATURES))[:, 1]
    roc = roc_auc_score(valid["isFraud"], scores)
    pr = average_precision_score(valid["isFraud"], scores)
    print(f"seed {seed:<3} ROC-AUC {roc:.4f}   PR-AUC {pr:.4f}")

evaluate(train, valid)

seed 42  ROC-AUC 0.9247   PR-AUC 0.5884


In [5]:
for seed in [1, 2]:
    evaluate(train, valid, seed)

seed 1   ROC-AUC 0.9213   PR-AUC 0.5836
seed 2   ROC-AUC 0.9254   PR-AUC 0.5941


## 1. Time features

In [6]:
from fraud.features import add_time_features

df = add_time_features(df)
train, valid, _ = time_split(df)

In [7]:
(train.groupby("hour")["isFraud"].mean() * 100).round(1)

hour
0      3.2
1      3.3
2      3.6
3      3.5
4      5.0
5      6.2
6      7.6
7      9.7
8     10.0
9      9.6
10     7.5
11     4.1
12     3.1
13     2.2
14     2.4
15     2.5
16     3.0
17     3.2
18     3.5
19     3.5
20     3.4
21     3.4
22     3.3
23     3.9
Name: isFraud, dtype: float64

In [8]:
(train.groupby("weekday")["isFraud"].mean() * 100).round(1)

weekday
0    3.8
1    3.6
2    3.7
3    3.7
4    3.2
5    3.3
6    3.4
Name: isFraud, dtype: float64

In [9]:
for seed in [42, 1, 2]:
    evaluate(train, valid, seed)

seed 42  ROC-AUC 0.9222   PR-AUC 0.5835
seed 1   ROC-AUC 0.9222   PR-AUC 0.5907
seed 2   ROC-AUC 0.9220   PR-AUC 0.5891


`hour` and `weekday` are computed from a single row, so nothing is learned from other rows and they cannot leak.
The fraud rate differs strongly by hour on train (2.2% to 10.0%), but adding both features does not improve the
model: mean PR-AUC over three seeds is 0.589 without them and 0.588 with them, well within the seed noise of about
0.01. They stay out of the main model and will be re-checked for logistic regression and the lite demo model.

## 2. Simple row features

In [10]:
from fraud.features import add_row_features

df = df.drop(columns=["hour", "weekday"])
df = add_row_features(df)
train, valid, _ = time_split(df)

In [11]:
(train.groupby("amount_3dec")["isFraud"].mean() * 100).round(2)

amount_3dec
False     2.57
True     11.24
Name: isFraud, dtype: float64

In [12]:
(train.groupby("ProductCD", observed=True)["amount_3dec"].mean() * 100).round(1)

ProductCD
C    90.4
H     0.0
R     0.0
S     0.0
W     0.0
Name: amount_3dec, dtype: float64

In [13]:
(train.groupby("has_identity")["isFraud"].mean() * 100).round(2)

has_identity
False    2.14
True     7.34
Name: isFraud, dtype: float64

In [14]:
CANDIDATES = ["amount_cents", "amount_3dec", "has_identity", "n_missing"]

for name in CANDIDATES:
    others = [c for c in CANDIDATES if c != name]
    print(name)
    for seed in [42, 1, 2]:
        evaluate(train.drop(columns=others), valid.drop(columns=others), seed)

amount_cents
seed 42  ROC-AUC 0.9233   PR-AUC 0.5898
seed 1   ROC-AUC 0.9205   PR-AUC 0.5892
seed 2   ROC-AUC 0.9230   PR-AUC 0.5840
amount_3dec
seed 42  ROC-AUC 0.9221   PR-AUC 0.5867
seed 1   ROC-AUC 0.9211   PR-AUC 0.5871
seed 2   ROC-AUC 0.9234   PR-AUC 0.5873
has_identity
seed 42  ROC-AUC 0.9199   PR-AUC 0.5884
seed 1   ROC-AUC 0.9206   PR-AUC 0.5905
seed 2   ROC-AUC 0.9224   PR-AUC 0.5865
n_missing
seed 42  ROC-AUC 0.9190   PR-AUC 0.5828
seed 1   ROC-AUC 0.9209   PR-AUC 0.5879
seed 2   ROC-AUC 0.9233   PR-AUC 0.5915


Four simple row features were tested one at a time: cents of the amount, a third decimal digit, an identity flag
and the number of missing fields. Some look strong on their own (a third decimal: 11.2% fraud vs 2.6%; identity
present: 7.3% vs 2.1%), but none improves the model: mean PR-AUC over three seeds is 0.587–0.588 against the
baseline 0.589, within the seed noise. The model already sees this information: a third decimal occurs only in
product `C` (90% of its amounts), and identity presence and missingness are visible as NaN in the original
columns. All four are dropped for LightGBM; they may still help logistic regression.

In [15]:
df = df.drop(columns=CANDIDATES)

## 3. Card start day

In [16]:
print(f"D1 mean: train {train['D1'].mean():.0f}, valid {valid['D1'].mean():.0f}")

D1 mean: train 88, valid 111


In [17]:
from fraud.features import add_card_start_day

df = add_card_start_day(df)
train, valid, _ = time_split(df)

for seed in [42, 1, 2]:
    evaluate(train, valid, seed)

seed 42  ROC-AUC 0.9235   PR-AUC 0.6029
seed 1   ROC-AUC 0.9246   PR-AUC 0.6017
seed 2   ROC-AUC 0.9235   PR-AUC 0.6014


`card_start_day` is the transaction day minus `D1`. `D1` grows with time (mean 88 on train vs 111 on valid);
Kaggle participants read it as days since the card was first used, so the difference stays the same for all
transactions of one card and helps the model recognise the card. This matters because labels are propagated:
after a chargeback, later transactions linked to the same account are also marked as fraud. Mean PR-AUC over
three seeds rises from 0.589 to 0.602 and every seed beats the best baseline seed, while ROC-AUC stays at 0.924.
The feature is kept.

## 4. Client key

Hypothesis: the card number is hidden, but a combination of the card columns (`card1`–`card6`) and the billing
address (`addr1`, `addr2`) is close to unique for one client. If so, the model can link a transaction to the same
client's earlier ones. This matters because labels are propagated: after a reported fraud, the client's later
transactions are also marked as fraud.

In [20]:
df[["card1", "card2", "card3", "card4", "card5", "card6", "addr1", "addr2"]].nunique()

card1    13553
card2      500
card3      114
card4        4
card5      119
card6        4
addr1      332
addr2       74
dtype: int64

In [21]:
CARD_COLS = ["card1", "card2", "card3", "card4", "card5", "card6"]

print("card1:", len(train[["card1"]].drop_duplicates()))
print("card1 + addr1:", len(train[["card1", "addr1"]].drop_duplicates()))
print("all card + addr:", len(train[CARD_COLS + ["addr1", "addr2"]].drop_duplicates()))
print("all + card_start_day:", len(train[CARD_COLS + ["addr1", "addr2", "card_start_day"]].drop_duplicates()))

card1: 12251
card1 + addr1: 34787
all card + addr: 37142
all + card_start_day: 168440


In [27]:
KEYS = {
    "card1": ["card1"],
    "card1 + addr1": ["card1", "addr1"],
    "all + card_start_day": CARD_COLS + ["addr1", "addr2", "card_start_day"],
    "P_emaildomain + all + card_start_day": CARD_COLS + ["addr1", "addr2", "card_start_day", "P_emaildomain"],
    "dist1 + all + card_start_day": CARD_COLS + ["addr1", "addr2", "card_start_day", "dist1"],
    "DeviceInfo + all + card_start_day": CARD_COLS + ["addr1", "addr2", "card_start_day", "DeviceInfo"],
}

for name, cols in KEYS.items():
    groups = train.groupby(cols, dropna=False)["isFraud"].agg(["size", "mean"])
    with_fraud = groups[(groups["size"] >= 2) & (groups["mean"] > 0)]
    print(f"{name}: {len(with_fraud)} groups with fraud, median fraud share inside {with_fraud['mean'].median():.2f}")

card1: 1299 groups with fraud, median fraud share inside 0.08
card1 + addr1: 2283 groups with fraud, median fraud share inside 0.13
all + card_start_day: 3682 groups with fraud, median fraud share inside 0.60
P_emaildomain + all + card_start_day: 3479 groups with fraud, median fraud share inside 1.00
dist1 + all + card_start_day: 3755 groups with fraud, median fraud share inside 0.75
DeviceInfo + all + card_start_day: 3334 groups with fraud, median fraud share inside 1.00


Counting distinct combinations on train: `card1` alone gives 12,251 groups, adding `addr1` gives 34,787, the other
card columns and `addr2` add little (37,142), and `card_start_day` splits much further (168,440). To check whether a
group is one client, we use how labels were set: after a reported fraud, the client's later transactions are also
marked as fraud, so a group that is one client should be mostly fraud once it contains fraud. Among groups with at
least two transactions and at least one fraud, the median fraud share is 0.08 for `card1`, 0.13 for `card1` + `addr1`
and 0.60 for the full key. The card and address columns alone mix many clients; with `card_start_day` the key is
much closer to one client.

In [28]:
shuffled = train.copy()
shuffled["isFraud"] = train["isFraud"].sample(frac=1, random_state=0).to_numpy()

for name, cols in KEYS.items():
    groups = shuffled.groupby(cols, dropna=False)["isFraud"].agg(["size", "mean"])
    with_fraud = groups[(groups["size"] >= 2) & (groups["mean"] > 0)]
    print(f"{name}: median fraud share inside {with_fraud['mean'].median():.2f}")

card1: median fraud share inside 0.06
card1 + addr1: median fraud share inside 0.08
all + card_start_day: median fraud share inside 0.20
P_emaildomain + all + card_start_day: median fraud share inside 0.25
dist1 + all + card_start_day: median fraud share inside 0.25
DeviceInfo + all + card_start_day: median fraud share inside 0.25


Smaller groups give a higher fraud share by chance alone, so the same check is repeated with `isFraud` shuffled:
group sizes stay the same, but any link between a client and fraud is gone. With shuffled labels the median share
is 0.06–0.25 for all keys. With real labels it is 0.08 for `card1` (almost chance), 0.60 for the full key, 0.75 with
`dist1` and 1.00 with `P_emaildomain` or `DeviceInfo`. The gap over chance grows from 0.02 to 0.75, so the email
domain and the device really help to separate clients; it is not just the smaller groups. This check only sees
different clients mixed in one group, not one client split across groups (for example, two devices), so the model
makes the final choice.

In [29]:
train["P_emaildomain"].notna().mean()

np.float64(0.8444934409541132)

The key for the model is the card columns, the address, `card_start_day` and `P_emaildomain`. The email domain is
filled for most transactions, unlike the device, and one client usually keeps one email, while a key with the
device could split a client who uses two devices.

In [30]:
KEY_COLS = CARD_COLS + ["addr1", "addr2", "card_start_day", "P_emaildomain"]

df["client_tx_before"] = df.groupby(KEY_COLS, dropna=False).cumcount()
train, valid, _ = time_split(df)

In [31]:
(train.groupby(train["client_tx_before"].clip(upper=5))["isFraud"].mean() * 100).round(2)

client_tx_before
0    2.77
1    3.62
2    3.79
3    3.96
4    4.18
5    5.58
Name: isFraud, dtype: float64

In [32]:
for seed in [42, 1, 2]:
    evaluate(train, valid, seed)

seed 42  ROC-AUC 0.9246   PR-AUC 0.5974
seed 1   ROC-AUC 0.9265   PR-AUC 0.6074
seed 2   ROC-AUC 0.9241   PR-AUC 0.6048


`client_tx_before` counts the client's earlier transactions (`cumcount` over the key in time order, so only the
past is used). On train the fraud rate grows with it, from 2.8% on a client's first transaction to 5.6% on the sixth
and later, likely because a caught client's later transactions are all labeled fraud. On valid, PR-AUC over seeds
42, 1, 2 is 0.5974, 0.6074 and 0.6048 (mean 0.603) against 0.6029, 0.6017 and 0.6014 (mean 0.602) without it:
seed 42 is lower, so the gain is within the seed noise. The data already has counting columns (`C1`–`C14`), which
may carry the same information. The feature is not kept.

In [33]:
df = df.drop(columns=["client_tx_before"])
train, valid, _ = time_split(df)

In [35]:
by_client = df.groupby(KEY_COLS, dropna=False)['TransactionAmt']
past_count = by_client.cumcount()
past_sum = by_client.cumsum() - df['TransactionAmt']
df['amount_vs_client_mean'] = df["TransactionAmt"] / (past_sum / past_count)
train, valid, _ = time_split(df)

In [36]:
ratio_group = train["amount_vs_client_mean"].clip(upper=5).round()
(train.groupby(ratio_group, dropna=False)["isFraud"].mean() * 100).round(2)

amount_vs_client_mean
0.0    3.63
1.0    4.44
2.0    4.21
3.0    4.25
4.0    3.80
5.0    4.38
NaN    2.77
Name: isFraud, dtype: float64

In [37]:
for seed in [42, 1, 2]:
    evaluate(train, valid, seed)

seed 42  ROC-AUC 0.9247   PR-AUC 0.5986
seed 1   ROC-AUC 0.9235   PR-AUC 0.5960
seed 2   ROC-AUC 0.9256   PR-AUC 0.6064


`amount_vs_client_mean` is the amount divided by the mean of the client's earlier amounts (past only; empty on the
client's first transaction). On train the fraud rate barely depends on it: 3.6-4.4% in every group, from less than
half of the usual amount to five times and more, against 2.8% on first transactions. A likely reason: after the
first caught fraud the client's later transactions are all labeled fraud, and the usual amount already includes
the fraudster's own purchases. On valid, PR-AUC over seeds 42, 1, 2 is 0.5986, 0.5960 and 0.6064 (mean 0.600)
against 0.602 without it, within the seed noise. The feature is not kept.

In [38]:
df = df.drop(columns=["amount_vs_client_mean"])
train, valid, _ = time_split(df)